# Train ResNet50 EuroSAT Classifier (GPU)
Run this on Google Colab with a free GPU (~15-25 min).

Goal: fine-tune the **ResNet50** backbone (the served app model is the weaker MobileNetV2) to ~98% EuroSAT test accuracy, then download it and re-run the Evros change-detection sweep locally:

```bash
.venv/bin/python -m change_detection.experiment_clf_scaling \
    --tiff-a data/evros_2021.tif --tiff-b data/evros_2025.tif \
    --gfw-lossyear results/evros/lossyear_50N_020E.tif \
    --model-path models/resnet50_eurosat_ft.h5
```

In [ ]:
# Step 1: Mount Google Drive (to save the model)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: Install dependencies
!pip install datasets -q
import tensorflow as tf
import numpy as np
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Step 3: Load EuroSAT from Hugging Face
from datasets import load_dataset
hf = load_dataset('giswqs/EuroSAT_RGB')

def load(split):
    ds = hf[split]
    imgs, lbls = [], []
    for i in range(len(ds)):
        imgs.append(np.array(ds[i]['image'], dtype=np.float32) / 255.0)
        lbls.append(ds[i]['label'])
    return np.array(imgs), np.array(lbls)

x_train, y_train = load('train')
x_val, y_val = load('validation')
x_test, y_test = load('test')
print(f'Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}')

In [ ]:
# Step 4: Build ResNet50 with ImageNet preprocessing INSIDE the graph
#
# IMPORTANT: the frozen ResNet50 expects mean-subtracted BGR input, so we
# register the same preprocessing function the local project uses
# (model_handler.resnet50_preprocess, namespace 'terra'). The saved model
# consumes RAW [0,1] RGB (the pipeline convention) and preprocesses itself.
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

@tf.keras.utils.register_keras_serializable('terra')
def resnet50_preprocess(x):
    return tf.keras.applications.resnet50.preprocess_input(x * 255.0)

inputs = tf.keras.Input((64, 64, 3))
x = layers.Lambda(resnet50_preprocess)(inputs)
base = tf.keras.applications.ResNet50(include_top=False, weights='imagenet',
                                      input_shape=(64, 64, 3))
x = base(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(10, activation='softmax')(x)
model = Model(inputs, out)
print('built:', model.count_params(), 'params')

In [ ]:
# Step 5: Data augmentation (same recipe as the original training)
# NOTE: value_range is REQUIRED on float [0,1] images -- without it this
# Keras version applies [0,255]-style brightness factors and blows values
# up to ~26, collapsing training to near-random (measured locally).
aug = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomBrightness(0.1, value_range=(0.0, 1.0)),
    layers.RandomContrast(0.1, value_range=(0.0, 1.0)),
])

def make_ds(x, y, batch_size=128, shuffle=False, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(5000)
    if augment:
        ds = ds.map(lambda img, lbl: (aug(img, training=True), lbl),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

ds_train = make_ds(x_train, y_train, shuffle=True, augment=True)
ds_val = make_ds(x_val, y_val)
ds_test = make_ds(x_test, y_test)

In [ ]:
# Step 6a: Phase 1 -- frozen backbone, train only the head (~2 min on GPU)
base.trainable = False
model.compile(optimizer=Adam(1e-3), loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.fit(ds_train, validation_data=ds_val, epochs=8, verbose=1,
          callbacks=[tf.keras.callbacks.EarlyStopping(
              monitor='val_accuracy', patience=3, restore_best_weights=True)])

In [ ]:
# Step 6b: Phase 2 -- unfreeze the LAST ResNet50 stage (conv5) + head,
# fine-tune at a low LR with augmentation (this is the ~98% push)
for layer in base.layers:
    layer.trainable = 'conv5' in layer.name
model.compile(optimizer=Adam(1e-4), loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.fit(ds_train, validation_data=ds_val, epochs=15, verbose=1,
          callbacks=[
              tf.keras.callbacks.EarlyStopping(
                  monitor='val_accuracy', patience=4, restore_best_weights=True),
              tf.keras.callbacks.ReduceLROnPlateau(
                  factor=0.5, patience=2, min_lr=1e-6),
          ])

In [ ]:
# Step 7: Evaluate on the held-out test split
test_loss, test_acc = model.evaluate(ds_test, verbose=1)
print(f'Test accuracy: {test_acc:.4f}')

In [ ]:
# Step 8: Save to Drive as resnet50_eurosat_ft.h5
import os
os.makedirs('/content/drive/MyDrive/eurosat_model', exist_ok=True)
model.save('/content/drive/MyDrive/eurosat_model/resnet50_eurosat_ft.h5')
!ls -lh /content/drive/MyDrive/eurosat_model/
print('Saved!')
print(f'Final test accuracy: {test_acc:.4f}')

## Done! Download and use locally

1. Download `resnet50_eurosat_ft.h5` from Drive (**Files** tab > `MyDrive/eurosat_model/`).
2. Place it in the project's `models/` folder.
3. Re-run the Evros change-detection sweep with it:

```bash
.venv/bin/python -m change_detection.experiment_clf_scaling \
    --tiff-a data/evros_2021.tif --tiff-b data/evros_2025.tif \
    --gfw-lossyear results/evros/lossyear_50N_020E.tif \
    --model-path models/resnet50_eurosat_ft.h5
```

The saved model embeds the registered `terra>resnet50_preprocess` function, so
the local pipeline (which imports `model_handler`) loads it with no extra steps.